In [26]:
from pathlib import Path
import pandas as pd

root = Path("..")
CSV_PATH    =  root / "Results/new_experiment_harness.csv"
OUT_PATH    = root / "ThesisPaper/tables/fb_results_table.tex"
CAP = 500_001

# ------------------------------------------------------------------
# Invalidity rule.
# A run is invalid if the full optimality-system residual norm is above
# tolerance — a stricter and more faithful "is this really a solution"
# measure than ||H_u|| alone, since it also catches state/adjoint
# violations.
# ------------------------------------------------------------------
def is_invalid(row) -> bool:
    return row["rk4_residual_norm"] > 1e-5 or row["fb_residual_norm"] > 1e-5


In [27]:
# ------------------------------------------------------------------
# Load + sort
# ------------------------------------------------------------------
import ast

df = pd.read_csv(CSV_PATH)
df = df.drop(columns=["m", "k", "x_d"])      # constants
df = df[~df["N"].isin([1000, 2000])]         # exclude per request
df = df.sort_values(by=["α", "N", "x₀"]).reset_index(drop=True)

# ------------------------------------------------------------------
# Format cells
# ------------------------------------------------------------------
def fmt_iter(n):
    return r"\textbf{---}" if n == CAP else f"{n:,}"

def fmt_time(t):
    if t < 60:
        return f"{t:.1f}s"
    if t < 3600:
        m, s = divmod(t, 60)
        return f"{int(m)}m {s:.1f}s"
    h, rem = divmod(t, 3600)
    m, s = divmod(rem, 60)
    return f"{int(h)}h {int(m)}m {s:.1f}s"

def fmt_sci(x):
    """Two-significant-digit scientific notation, LaTeX-friendly."""
    if pd.isna(x):
        return r"\text{NaN}"
    s = f"{x:.2e}"            # e.g. '1.46e+08' or '3.03e-11'
    mant, exp = s.split("e")
    exp = int(exp)
    return rf"${mant}\!\times\!10^{{{exp}}}$"

def last_step(s):
    """Pull the last entry from a string-encoded list of step lengths."""
    try:
        lst = ast.literal_eval(s)
        return float(lst[-1])
    except Exception:
        return float("nan")

df.head()

,rk4_time,rk4_iterations,rk4_g_norm,rk4_step_lenghts,rk4_residual_norm,fb_time,fb_iterations,fb_g_norm,fb_step_lenghts,fb_residual_norm,α,N,x₀
0,3.201805,4541,1.457965e+08,"[3.3780247916274465e-22, 3.4088147112011706e-2...",1.457965e+08,135.312588,500001,6.079061e+02,"[1.0e-6, 1.0e-6, 1.0e-6]",6.079061e+02,1.0,100,"[1.0, 1.0]"
1,5.189366,5838,3.031247e-11,"[0.0007196384946861832, 0.00031687616000319207...",3.031252e-11,170.498302,500001,2.432040e+02,"[1.0e-6, 1.0e-6, 1.0e-6]",2.432040e+02,1.0,100,"[1.0, 2.0]"
2,45.870124,41767,1.042777e+02,"[7.497636093040097e-14, 7.497725854016209e-14,...",1.042777e+02,0.762274,1831,2.903322e-11,"[0.0008708129882838586, 0.0008708091049272765,...",2.903322e-11,1.0,100,"[1.0, 3.0]"
3,2.818804,3131,4.710500e-10,"[0.001456434124353935, 0.034049813213364685, 1...",4.710500e-10,165.452117,500001,1.107928e+03,"[1.0e-6, 1.0e-6, 1.0e-6]",1.107928e+03,1.0,100,"[1.0, 4.0]"
4,58.599700,38720,3.703991e+07,"[6.60433819166662e-18, 2.561489468310981e-17, ...",3.703991e+07,250.600301,500001,1.548927e+03,"[1.0e-6, 1.0e-6, 1.0e-6]",1.548927e+03,1.0,200,"[1.0, 1.0]"


In [28]:
# ------------------------------------------------------------------
# Build rows manually so we can prepend \rowcolor
# ------------------------------------------------------------------
# Per-column labels (without the "RK4 " / "FB " prefix — those become a
# grouped header band above).
col_labels = [
    r"$\alpha$", r"$N$", r"$\mathbf{x_0}$",
    "iters", "time", r"$\|H_u\|$", r"$\|r\|$", r"$\theta_{\mathrm{last}}$",
    "iters", "time", r"$\|H_u\|$", r"$\|r\|$", r"$\theta_{\mathrm{last}}$",
]
# Vertical separator between the RK4 block (cols 4-8) and the FB block (cols 9-13).
col_spec = "r r c r r r r r|r r r r r"

# Two-row header: a banded "RK4" / "FB" row, then the column labels.
# The `c|` on the RK4 multicolumn carries the vertical rule through the
# header band so the separator is unbroken from top to bottom.
header_block = "\n".join([
    r"\toprule",
    r"\multicolumn{3}{c}{} & \multicolumn{5}{c|}{\textbf{FB RK4}} "
    r"& \multicolumn{5}{c}{\textbf{FB Euler}} \\",
    r"\cmidrule(lr){4-8} \cmidrule(lr){9-13}",
    " & ".join(col_labels) + r" \\",
    r"\midrule",
])

rows = []
for _, r in df.iterrows():
    cells = [
        f"{r['α']}",
        f"{r['N']}",
        f"{r['x₀']}",
        fmt_iter(r["rk4_iterations"]),
        fmt_time(r["rk4_time"]),
        fmt_sci(r["rk4_g_norm"]),
        fmt_sci(r["rk4_residual_norm"]),
        fmt_sci(last_step(r["rk4_step_lenghts"])),
        fmt_iter(r["fb_iterations"]),
        fmt_time(r["fb_time"]),
        fmt_sci(r["fb_g_norm"]),
        fmt_sci(r["fb_residual_norm"]),
        fmt_sci(last_step(r["fb_step_lenghts"])),
    ]
    line = " & ".join(cells) + r" \\"
    if is_invalid(r):
        line = r"\rowcolor{invalidRow} " + line
    rows.append(line)

# ------------------------------------------------------------------
# Emit longtable
# ------------------------------------------------------------------
caption = (
    r"Forward--backward results across the $(N,\alpha,\mathbf{x_0})$ grid. "
    r"\textbf{---} indicates the $500{,}000$-iteration limit was reached "
    r"without meeting tolerance. Rows shaded in red are runs whose "
    r"final optimality-system residual norm $\|r\|$ exceeded $10^{-5}$ "
    r"--- i.e.\ the iterate is not a discrete solution. $\|H_u\|$ is "
    r"shown alongside as the historical control-equation diagnostic, "
    r"and $\theta_{\mathrm{last}}$ is the final BB step length, whose "
    r"floating-point underflow drives the failure (see "
    r"Section~\ref{sec:fb-collapse})."
)

body = "\n".join([
    # 13 columns; shrink to \scriptsize + tight \tabcolsep so it fits
    # \textwidth (\resizebox does not work with longtable since it splits
    # across pages).
    r"\begingroup",
    r"\scriptsize",
    r"\setlength{\tabcolsep}{1.5pt}",
    r"\begin{longtable}{" + col_spec + "}",
    r"\caption{" + caption + r"}\label{tab:fb-results} \\",
    header_block,
    r"\endfirsthead",
    header_block,
    r"\endhead",
    r"\midrule \multicolumn{" + str(len(col_labels)) + r"}{r}{\textit{Continued on next page}} \\",
    r"\endfoot",
    r"\bottomrule",
    r"\endlastfoot",
    *rows,
    r"\end{longtable}",
    r"\endgroup",
])

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(OUT_PATH).write_text(body)
print(f"wrote {OUT_PATH} ({len(rows)} rows, {sum(is_invalid(r) for _, r in df.iterrows())} flagged)")


wrote ../ThesisPaper/tables/fb_results_table.tex (48 rows, 20 flagged)


In [29]:
rows

['\\rowcolor{invalidRow} 1.0 & 100 & [1.0, 1.0] & 4,541 & 3.2s & $1.46\\!\\times\\!10^{8}$ & $1.46\\!\\times\\!10^{8}$ & $3.47\\!\\times\\!10^{-23}$ & \\textbf{---} & 2m 15.3s & $6.08\\!\\times\\!10^{2}$ & $6.08\\!\\times\\!10^{2}$ & $1.00\\!\\times\\!10^{-6}$ \\\\',
 '\\rowcolor{invalidRow} 1.0 & 100 & [1.0, 2.0] & 5,838 & 5.2s & $3.03\\!\\times\\!10^{-11}$ & $3.03\\!\\times\\!10^{-11}$ & $3.08\\!\\times\\!10^{-4}$ & \\textbf{---} & 2m 50.5s & $2.43\\!\\times\\!10^{2}$ & $2.43\\!\\times\\!10^{2}$ & $1.00\\!\\times\\!10^{-6}$ \\\\',
 '\\rowcolor{invalidRow} 1.0 & 100 & [1.0, 3.0] & 41,767 & 45.9s & $1.04\\!\\times\\!10^{2}$ & $1.04\\!\\times\\!10^{2}$ & $1.63\\!\\times\\!10^{-18}$ & 1,831 & 0.8s & $2.90\\!\\times\\!10^{-11}$ & $2.90\\!\\times\\!10^{-11}$ & $5.49\\!\\times\\!10^{-5}$ \\\\',
 '\\rowcolor{invalidRow} 1.0 & 100 & [1.0, 4.0] & 3,131 & 2.8s & $4.71\\!\\times\\!10^{-10}$ & $4.71\\!\\times\\!10^{-10}$ & $1.20\\!\\times\\!10^{-5}$ & \\textbf{---} & 2m 45.5s & $1.11\\!\\times\\!